In [2]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import io
import pandas as pd

# ── STEP 1: Export all data sheets ───────────────────────────
with pd.ExcelWriter("MedSupply_Report.xlsx", engine="openpyxl") as writer:
    df_hospital.to_excel(writer,  sheet_name="By Hospital Type", index=False)
    df_products.to_excel(writer,  sheet_name="Product Ranking",  index=False)
    q1_commissions.to_excel(writer, sheet_name="Q1 Commissions", index=False)
    df_contracts.to_excel(writer, sheet_name="Contract Risk",    index=False)
    df.to_excel(writer,           sheet_name="Raw Data",         index=False)

# ── STEP 2: Open and format ───────────────────────────────────
wb = load_workbook("MedSupply_Report.xlsx")

# ── STEP 3: Executive Summary sheet ──────────────────────────
ws_summary = wb.create_sheet("Executive Summary", 0)

# Colors
navy_fill   = PatternFill(start_color="1F4E79", fill_type="solid")
white_font  = Font(color="FFFFFF", bold=True, size=14)
title_font  = Font(color="FFFFFF", bold=True, size=18)
value_font  = Font(bold=True, size=13)
label_font  = Font(color="595959", size=12)

# Title
ws_summary["B2"] = "MEDSUPPLY MÉXICO — EXECUTIVE REPORT"
ws_summary["B2"].font   = title_font
ws_summary["B2"].fill   = navy_fill
ws_summary["B2"].alignment = Alignment(horizontal="center")
wb.active = ws_summary
ws_summary.merge_cells("B2:E2")

# KPI rows
kpis = [
    ("Total Revenue (MXN)",     f"${df['venta_total'].sum():,.0f}"),
    ("Gross Margin (MXN)",      f"${df['margen_bruto'].sum():,.0f}"),
    ("Average Margin %",        f"{df['margen_pct'].mean():.1f}%"),
    ("Total Commissions (MXN)", f"${df['comision_mxn'].sum():,.0f}"),
    ("Total Transactions",      f"{len(df):,}"),
    ("Active Contracts",        f"{df['contrato_activo'].sum():,}"),
    ("Inactive Contracts",      f"{(df['contrato_activo']==0).sum():,}"),
]

for i, (label, value) in enumerate(kpis, start=4):
    ws_summary[f"B{i}"] = label
    ws_summary[f"C{i}"] = value
    ws_summary[f"B{i}"].font = label_font
    ws_summary[f"C{i}"].font = value_font
    ws_summary[f"C{i}"].alignment = Alignment(horizontal="right")
    # Alternating row colors
    if i % 2 == 0:
        fill = PatternFill(start_color="DEEAF1", fill_type="solid")
        ws_summary[f"B{i}"].fill = fill
        ws_summary[f"C{i}"].fill = fill

# Column widths
ws_summary.column_dimensions["B"].width = 30
ws_summary.column_dimensions["C"].width = 20

# ── STEP 4: Format Q1 Commissions sheet ──────────────────────
ws_comm = wb["Q1 Commissions"]

gold_fill   = PatternFill(start_color="FFD700", fill_type="solid")
green_fill  = PatternFill(start_color="C6EFCE", fill_type="solid")
red_fill    = PatternFill(start_color="FFC7CE", fill_type="solid")
header_fill = PatternFill(start_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

# Format headers
for cell in ws_comm[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

# Format rows by ranking
for row in ws_comm.iter_rows(min_row=2, max_row=ws_comm.max_row):
    # Ranking column — adjust index based on your query result columns
    try:
        ranking = row[-1].value  # Last column = ranking
        if ranking == 1:
            fill = gold_fill
        elif ranking in [2, 3]:
            fill = green_fill
        elif ranking == 10:
            fill = red_fill
        else:
            fill = PatternFill(fill_type=None)

        for cell in row:
            cell.fill = fill
    except:
        pass

# Auto-width all columns in commissions sheet
for col in ws_comm.columns:
    max_len = max(len(str(cell.value or "")) for cell in col) + 4
    ws_comm.column_dimensions[get_column_letter(col[0].column)].width = max_len

# Add totals row
last_row = ws_comm.max_row + 1
ws_comm[f"A{last_row}"] = "TOTAL"
ws_comm[f"A{last_row}"].font = Font(bold=True)

# ── STEP 5: Auto-width for all other sheets ───────────────────
for sheet_name in ["By Hospital Type", "Product Ranking", "Contract Risk"]:
    ws = wb[sheet_name]
    # Format headers
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center")
    # Auto-width
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col) + 4
        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len

# ── STEP 6: Add dashboard image ──────────────────────────────
# (runs after you create the figure with plt)
# ws_dash = wb.create_sheet("Dashboard", 1)
# img = XLImage(buffer)
# ws_dash.add_image(img, "A1")

# ── STEP 7: Save ─────────────────────────────────────────────
wb.save("MedSupply_Report.xlsx")
print("✅ Professional Excel report generated: MedSupply_Report.xlsx")

OSError: [Errno 30] Read-only file system: 'MedSupply_Report.xlsx'